# Same-host FP64 paper comparisons

Start a fresh Kaggle notebook with an **NVIDIA GPU** and **Internet enabled**, with at least 5 GiB free. The CUDA toolkit must support that GPU. CPU/GPU identification, topology, memory, driver/toolchain versions, and the detected CUDA architecture are recorded. This run compares CLQR, Vanroye, Yang's factor graph, original Laine, and corrected Laine–Tomlin on identical fixtures. It includes native CUDA and device-resident JAX timings, FP64 regression tests, and all four Compute Sanitizer checks.

The sweeps use $m=n/2$, $p_s=n/4$, $p_m=n/8$: every power-of-two horizon from 32 through 32768 crossed with every $n$ in 8,16,24,32,48,64 (66 cases), with no skipped pairs. Large cases can take substantially longer or exceed available memory. CSVs retain timings, primal/dual infinity-norm errors against the known optimum, and available original KKT residuals. Numerical failures are data, not reasons to stop other measurements.

Each backend runs once per case by default, with one untimed warmup solve and at least one second of cumulative timed solves (separately for prepared solves and native setup-plus-solve). Progress shows the backend, case number/total, dimensions, current phase, and saved-case count. Long commands print a heartbeat every 30 seconds; this confirms the process is still running, not that a solve has finished.

In [ ]:
import os
from pathlib import Path
import subprocess
import tempfile

# Editable defaults; pre-existing environment overrides take precedence.
os.environ.setdefault("CLQR_RUN_CPU", "1")  # native CPU timing sweep, not reference checks
os.environ.setdefault("CLQR_RUN_EXTERNAL", "1")  # 0 skips all external methods
for method in ("VANROYE", "YANG", "LAINE", "CORRECTED_LAINE"):
    os.environ.setdefault("CLQR_RUN_" + method, os.environ["CLQR_RUN_EXTERNAL"])
os.environ.setdefault("CLQR_RUN_JAX", "1")
os.environ.setdefault("CLQR_RUN_JAX_CPU", os.environ["CLQR_RUN_JAX"])
os.environ.setdefault("CLQR_RUN_JAX_GPU", os.environ["CLQR_RUN_JAX"])  # CUDA on this notebook
os.environ.setdefault("CLQR_RUN_TESTS", "1")
os.environ.setdefault("CLQR_RUN_SANITIZERS", "1")
os.environ.setdefault("CLQR_RUN_ORIGINAL_TABLE", "1")
os.environ.setdefault("CLQR_BENCHMARK_SECONDS", "1")  # cumulative measured time per timing phase
os.environ.setdefault("CLQR_BENCHMARK_ROUNDS", "1")
os.environ.setdefault("CLQR_PROGRESS_INTERVAL_SECONDS", "30")

# Use an uploaded snapshot if present; otherwise fetch current main once.
revision = os.environ.get("CLQR_REVISION", "main")
work = Path("/kaggle/working")
source = Path(tempfile.mkdtemp(prefix="clqr-source-", dir=work))
url = "https://github.com/joaospinto/constrained_lqr_elimination.git"
bundles = list(Path("/kaggle/input").rglob("clqr-source.bundle"))
if len(bundles) > 1:
    raise RuntimeError("Multiple clqr-source.bundle files found; attach only the snapshot to test")

subprocess.run(["git", "init", "-q", str(source)], check=True)
subprocess.run(["git", "-C", str(source), "remote", "add", "origin", url], check=True)
if bundles:
    print("Using source snapshot:", bundles[0], flush=True)
    subprocess.run(["git", "-C", str(source), "fetch", str(bundles[0]), revision], check=True)
else:
    subprocess.run(["git", "-C", str(source), "fetch", "--depth=1", "--filter=blob:none",
                    "origin", revision], check=True)
subprocess.run(["git", "-C", str(source), "-c", "advice.detachedHead=false",
                "checkout", "--detach", "FETCH_HEAD"], check=True)
print("Source revision:", subprocess.check_output(
    ["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip(), flush=True)

# Streams progress, archives results even on failure, then cleans its private build cache.
result = subprocess.run(["python3", "-u", str(source / "scripts/notebook_paper.py"),
                         "--work-dir", str(work)])
print("Notebook driver exit code:", result.returncode)


Optionally attach `clqr-source.bundle` as a Kaggle dataset for an exact source snapshot; without it, the notebook uses current `origin/main`. Internet is still needed for pinned dependencies. Download the printed `paper-results.zip` path from Kaggle's output browser. It includes the unified `measurements.csv`, individual rounds, summary, machine details (`platform.txt` and `gpu.csv`), and logs. The notebook removes only its own downloaded dependencies and generated build cache; results and the source checkout remain. No paper numbers are replaced automatically, and a Blackwell run is not labeled as P100.